In [1]:
import pandas as pd
import numpy as np
import os

from surprise import SVD,Dataset, Reader
from surprise import accuracy
from surprise.model_selection import train_test_split


In [2]:
import json

In [3]:
PATH_MOVIELENS = os.path.join('..','dataset','MovieLens')


In [4]:
movieId_lookup = pd.read_csv('../dataset/processed/movieId_lookup.csv')
movieId_lookup.shape

(3037, 3)

In [5]:
movielens_movies = pd.read_csv(os.path.join(PATH_MOVIELENS,'movies.csv'))
# filter movies which only exist in movieLookup table
movielens_movies = movielens_movies[movielens_movies['movieId'].isin(movieId_lookup['movieLens_movieId'])].reset_index(drop=True)

movielens_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,14,Nixon (1995),Drama
2,15,Cutthroat Island (1995),Action|Adventure|Romance
3,16,Casino (1995),Crime|Drama
4,18,Four Rooms (1995),Comedy


In [6]:
movielens_movies.shape

(3037, 3)

In [7]:
ratings = pd.read_csv(os.path.join(PATH_MOVIELENS,'ratings.csv'))
# filter ratings which movie Id exists in movieLookup table
ratings_filtered = ratings[ratings['movieId'].isin(movieId_lookup['movieLens_movieId'])].reset_index(drop=True)
ratings_filtered.head(2)

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,899,3.5,1147868510


In [8]:
ratings_filtered.shape

(10861376, 4)

In [9]:
ratings_filtered['movieId'].nunique()

3021

In [10]:
ratings_filtered['userId'].nunique()

162524

In [67]:
 5.000000e-01

0.5

In [11]:
ratings['rating'].describe()

count    2.500010e+07
mean     3.533854e+00
std      1.060744e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64

In [12]:
ratings_filtered.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [13]:
ratings = ratings.drop(columns='timestamp')

### SVD model

Only select movies ratings which are present in both movieLens and TMBD movie dataset

model building

In [70]:
rating_sample = ratings_filtered.sample(n = 250000,random_state=97)

In [71]:
# create reader
reader = Reader(
        rating_scale=(0.5,5)
        )


In [72]:
data = Dataset.load_from_df(
    df = rating_sample[['userId','movieId','rating']],
    reader=reader
    )

In [73]:

trainset,testset = train_test_split(data,test_size=0.2,random_state=42)



In [74]:
model = SVD(n_factors=100,n_epochs=25,lr_all = 0.009,reg_all=0.02,verbose=True)

In [75]:
# train model on train dataset only
model.fit(trainset)

Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Processing epoch 20
Processing epoch 21
Processing epoch 22
Processing epoch 23
Processing epoch 24


In [76]:
# predict the result for unseen dataset
predictions = model.test(testset)

In [77]:
# overview the prediction result
predictions[0]

Prediction(uid=6344, iid=1961, r_ui=4.0, est=np.float64(3.929188290212658), details={'was_impossible': False})

In [78]:
print("Model Evaluation")
print('rmse : ',accuracy.rmse(predictions))
print('mse : ',accuracy.mse(predictions))
print('mae : ',accuracy.mae(predictions))

Model Evaluation
RMSE: 0.9493
rmse :  0.9493225356143641
MSE: 0.9012
mse :  0.9012132766252856
MAE:  0.7349
mae :  0.7349018285432708


In [79]:
# predict the rating of movie id 'I' for user no 'U'
model.predict(uid=97,iid=5378)

Prediction(uid=97, iid=5378, r_ui=None, est=np.float64(3.2829997305286063), details={'was_impossible': False})

In [80]:
testset_df =pd.DataFrame(columns=['userId','movieId','rating'],data=(np.array(testset)))
testset_df['userId'] = testset_df['userId'].astype('int')
testset_df['movieId'] = testset_df['movieId'].astype('int64')



In [81]:

testset_df.query('userId ==97').sort_values(by='rating' ,ascending=False)

,userId,movieId,rating
3307,97,5378,2.5


In [82]:
model.predict(uid=5,iid=296)

Prediction(uid=5, iid=296, r_ui=None, est=np.float64(4.31477049146296), details={'was_impossible': False})

In [83]:
# save svd model in binary pickle format for future usages.
import pickle

pickle.dump(model,open('../artifact/svd_model.pkl','wb'))

### prediction demo

In [84]:
# user_id = 645
# random user
user_id = ratings_filtered['userId'].unique()[np.random.randint(1,162524)]

# get all watched movies for user U
watched_movies = set(ratings_filtered[ratings_filtered['userId']==user_id]['movieId'])

# get all movies which exist in dataset.
all_movies = set(movieId_lookup['movieLens_movieId'])

# find all non watched movies by users.
unwatched_movies =  all_movies - watched_movies

predictions = []
# calculate the prediction rating for user U for unwatched movie uM
for movie in unwatched_movies:
    pred_rating = model.predict(user_id,movie).est
    predictions.append([movie,pred_rating])
   

# sort the predicted rating by highly rated movie prediction
predictions.sort(key=lambda x : x[1],reverse=True)

movie_lookup = dict(zip(movielens_movies['movieId'],movielens_movies['title']))

for movieId, rating in predictions[:10]:
    print(movie_lookup[movieId],' : ',rating)

One Flew Over the Cuckoo's Nest (1975)  :  4.253267246770918
Apocalypse Now (1979)  :  4.235958151441624
It Happened One Night (1934)  :  4.226346622163668
Fight Club (1999)  :  4.212189354086284
Inception (2010)  :  4.201722722924654
Goodfellas (1990)  :  4.201349323572033
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)  :  4.197599273246513
Some Like It Hot (1959)  :  4.172173785547581
Rebecca (1940)  :  4.166519797483818
Midnight Cowboy (1969)  :  4.164186285028516


select the sample users which are use in demonation of content based filtering.


In [113]:
# find the how much movies user watch
user_rating_size = ratings_filtered.groupby('userId').size().reset_index()
# select users randomly which watched more than 200 movies 
user_rating_size.columns = ['userId','size']
userId_recommandation = user_rating_size[user_rating_size['size']>200].sample(50)['userId'].tolist()


In [120]:
# filter the user which are in userId recommandation list and then find their movieId (watched)
user_watched_movies = (ratings_filtered[ratings_filtered['userId'].isin(userId_recommandation)]
                        .groupby('userId')['movieId'].apply(set)
                      )

In [123]:
user_watched_movies.to_json('../dataset/processed/userWatched_movies.json')